In [ ]:
print("Started................")

In this Notebook, 

I am trying to use BDR (Bi-Dimensional Regression) on our Sketchmaps.
Rather than being a statistic, BDR is more like a framework, where multiple statistical measures are used. #

These statistical measures are as follows:

1. r -> Correlation coefficient (This shows how the map overall looks like)
2. R^2 -> Covariance (How well is everything placed in the sketchmap)
3. DI -> Distortion Index (from 0 to 100, shows how messy a drawing is, or how distorted is a sketchmap.)
4. Alpha 1 -> Horizontal shift (did the person shift all the building in the east?) Negative value indicates West
5. Alpha 2 -> Vertical shift (did the person shift all the buildings in the North? ) Negative value indicates south
6. Theta -> Rotation Angle (shows whether the whole sketchmap is rotated by a certail angle)
7. phi -> scale (shows whether the things in sketchmap are larger or smaller than the basemap.)

These measures constitutes BDR (Bi - Dimensional Regression).

BDR always works on 2 points or in this case polygons (which are also being denoted by points in this case). 

Now, these measures are ran individualy on a pair (one point in Basemap and another in sketchmap) and then all of these values are averaged by all of the buildings in the sketchmap. 




In [ ]:
import geopandas as gpd
import json
import matplotlib.pyplot as plt

In [ ]:
bsm_feat = gpd.read_file("/kaggle/input/datasets/ajayuchana/stechmap-bdr-test/basemap.jpg.geojson")
bsm_landmarks  = bsm_feat[bsm_feat['feat_type'] == 'Landmark']


print(bsm_landmarks)
bsm_landmarks.plot()

In [ ]:
skm_feat = gpd.read_file("/kaggle/input/datasets/ajayuchana/stechmap-bdr-test/sketchMap1.JPG.geojson")



skm_landmarks = skm_feat[skm_feat['feat_type'] == 'Landmark']
print(skm_landmarks)
skm_landmarks.plot()

In [ ]:
with open('/kaggle/input/datasets/ajayuchana/alignm/alignment.json', 'r') as file:
    align = json.load(file)


print(align['sketchMap1.JPG'])

sketchmap1_align = align['sketchMap1.JPG']

I have three files now: 
        1. Basemap GeoJSON file
        2. Sketchmap GeoJSON file
        3. Alignment JSON file

Alignment file contains how I the landmarks in Basemap are connected to Sketchmap. 

So, there is a key in Sketchmap GeoJSON 'sid' which is mentioned in Alignment file and also an id of Basemap.

This will help us match Landmarks in these files. 

In [ ]:
from skimage import transform 
import numpy as np


base_pts = []
sketch_pts = []
pair_labels = []

for key, value in sketchmap1_align.items():
    if key == "checkAlignnum":
        continue

    base_indices = value['BaseAlign']['0']
    sketch_sids = value['SketchAlign']['0']

    s_id = sketch_sids[0]

    
    s_feat = skm_landmarks[skm_landmarks['sid'] == s_id]
    b_feat = bsm_landmarks[bsm_landmarks['id'].isin(base_indices)]

    if not s_feat.empty and not b_feat.empty:
        s_centroid = s_feat.geometry.centroid.values[0]
        sketch_pts.append([s_centroid.x, s_centroid.y])

        b_centroids = b_feat.geometry.representative_point()
        mean_x = b_centroids.x.mean()
        mean_y = b_centroids.y.mean()
        base_pts.append([mean_x, mean_y])


X = np.array(base_pts)
Y = np.array(sketch_pts)



tform = transform.SimilarityTransform()
tform.estimate(X,Y)

print("BDR Complete")
print(f" {len(X)} valid alignment pairs.")
print(f" Scale: {tform.scale:.4f}")
print(f" Rotation: {np.degrees(tform.rotation):.2f}")

In the cell above, 

I have taken three lists: 
1. base_pts (stores the points from basemap)
2. sketch_pts (stores the points from sketchmap) and 
3. pair_labels (stores the pairs which are common in basemap and sketchmap)


Then, extract the pairs from the alignment file. 
Where, the common landmarks are matched using, 

'id' from basemap
'sid' from sketchmap

Now, we extract where both id and sid are not empty, giving us the common polygons. 
After this, I take the building id from basemap and sketchmap (they are already aligned) and draw a centroid using the geometry of the buildings in sketchmap.

For basemap, we use representative point, which makes sure the centre point of our building is always inside the building and not out. Since, the buildings in the basemap can have complex shapes, which might include a lot of corners or vertices. 


Then, we store all those points from basemap in X (ground truth), which is the independent variable and 
we store all the points from sketchmap in Y (drawn by user), is the dependent variable. 

WE store them into numpy array, because that is the acceptable format for BDR.


Then we use the SimilarityTransfrom function, which matches how well these two maps match, but movement, rotation and scaling the basemap is allowed to make it fit the sketchmap.


After that, the function find the best rotation, the best scale and the best shift, which results in the smallest possible distrance betweeen the ground truth and the corresponding sketchmaps.

# BDR

In [ ]:
import matplotlib.pyplot as plt
from skimage import transform
from shapely.affinity import affine_transform
import numpy as np

# --- 1. DATA PROCESSING & SYNCHRONIZATION ---
base_pts = []
sketch_pts = []
pair_labels = []

for key, value in sketchmap1_align.items():
    if key == "checkAlignnum": continue
    
    b_indices = value['BaseAlign']['0']
    s_sid = value['SketchAlign']['0'][0]
    
    b_feats = bsm_landmarks[bsm_landmarks['id'].isin(b_indices)]
    s_feat = skm_landmarks[skm_landmarks['sid'] == s_sid]
    
    if not s_feat.empty and not b_feats.empty:
        # Use representative_point to ensure dots stay inside building footprints
        b_point = b_feats.geometry.representative_point()
        base_pts.append([b_point.x.mean(), b_point.y.mean()])
        
        s_point = s_feat.geometry.representative_point().values[0]
        sketch_pts.append([s_point.x, s_point.y])
        pair_labels.append(s_sid) 

X = np.array(base_pts)
Y = np.array(sketch_pts)

# --- 2. BIDIMENSIONAL REGRESSION ---
tform = transform.SimilarityTransform()
tform.estimate(X, Y)
predicted_Y = tform(X)

# --- 3. GEOMETRY TRANSFORMATION ---
# Move the Real Landmarks into Sketch Space so they overlap for the plot
m = tform.params
matrix = [m[0,0], m[0,1], m[1,0], m[1,1], m[0,2], m[1,2]]
bsm_transformed = bsm_landmarks.copy()
bsm_transformed['geometry'] = bsm_transformed['geometry'].apply(lambda x: affine_transform(x, matrix))

# IMPORTANT: Strip CRS to prevent "aspect must be finite" error
bsm_transformed.crs = None
skm_landmarks.crs = None

# --- 4. PLOTTING ---
fig, ax = plt.subplots(figsize=(14, 12))
ax.set_aspect('equal')

# Plot Polygons
bsm_transformed.plot(ax=ax, color='lightgray', edgecolor='gray', alpha=0.4, label='Real World (Projected)')
skm_landmarks.plot(ax=ax, color='orange', edgecolor='darkorange', alpha=0.3, label='User Sketch')

# Plot Dots
# We plot the first one with a label for the legend, then loop the rest
ax.scatter(predicted_Y[:, 0], predicted_Y[:, 1], color='blue', s=60, marker='o', 
           label='Truth Anchor (Blue)', zorder=5)
ax.scatter(Y[:, 0], Y[:, 1], color='red', s=60, marker='x', 
           label='Sketch Anchor (Red)', zorder=5)

# Plot Labels and Connectors
for i in range(len(Y)):
    label = pair_labels[i]
    
    # Label the Blue Dot
    ax.text(predicted_Y[i, 0], predicted_Y[i, 1], f" {label}", 
            color='blue', fontsize=8, fontweight='bold', va='bottom')
    
    # Label the Red Dot
    ax.text(Y[i, 0], Y[i, 1], f" {label}", 
            color='red', fontsize=8, fontweight='bold', va='top')
    
    # Connector line (Error Vector)
    line, = ax.plot([predicted_Y[i, 0], Y[i, 0]], [predicted_Y[i, 1], Y[i, 1]], 
                    color='black', linestyle=':', alpha=0.4, zorder=4)

# Formatting
plt.title(f"Map Alignment\nScale Factor: {tform.scale:.3f} | Rotation: {np.degrees(tform.rotation):.2f}°", 
          fontsize=14, pad=20)

# Manually handle legend because geopandas.plot and ax.scatter work differently
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='lightgray', lw=4, label='Real Landmarks (Transformed)'),
    Line2D([0], [0], color='orange', lw=4, alpha=0.5, label='Sketch Landmarks'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Truth Location'),
    Line2D([0], [0], marker='x', color='red', markersize=10, label='Sketch Location'),
    Line2D([0], [0], color='black', linestyle=':', label='Displacement (Error)')
]
ax.legend(handles=legend_elements, loc='upper right', frameon=True, shadow=True)

ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
print(predicted_Y)

## DI (Distortion Index)

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(Y, predicted_Y)
distortion_index = 100 * np.sqrt(1 - r2)
print(f"Distortion Index: {distortion_index:.2f}")

## Translation

In [ ]:
alpha1 = tform.translation[0]
alpha2 = tform.translation[1]
print(f"Alpha 1 (X-Shift): {alpha1:.2f}")
print(f"Alpha 2 (Y-Shift): {alpha2:.2f}")

## Correlation Coefficient (r)

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(Y, predicted_Y)
r = np.sqrt(r2)
print(f"Bidimensional Correlation (r): {r:.4f}")

## Rotation 

In [ ]:
# Convert radians to degrees
rotation_deg = np.degrees(tform.rotation)
print(f"Rotation (Theta): {rotation_deg:.2f}°")

# GARDONY BASED - MEASURES

In [ ]:
print("Clément")